# PINN Project 2: 1D Burgers' Equation (Time-Dependent)
## Physics-Informed Neural Network — From Steady-State to Time-Dependent

### Problem Description
The **Burgers' equation** is a fundamental PDE in fluid mechanics:

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

where:
- $u(x, t)$ is the velocity field
- $\nu = 0.01/\pi$ is the viscosity coefficient
- Domain: $x \in [-1, 1]$, $t \in [0, 1]$

**Boundary and Initial Conditions:**
- $u(x, 0) = -\sin(\pi x)$ (initial condition)
- $u(-1, t) = 0$ (left boundary)
- $u(1, t) = 0$ (right boundary)

### What's new compared to Project 1?
- **2 inputs**: (x, t) instead of just x
- **Time-dependent**: the solution evolves over time
- **Nonlinear**: the $u \cdot \partial u/\partial x$ term makes this much harder
- **We combine data + physics**: use some initial/boundary data along with PDE


In [1]:
# ============================================================
# STEP 1: Import Libraries
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cpu


## Step 2: Define the Network

Now the network takes **two inputs**: (x, t) → u(x, t)

We also add a technique called **input normalization** — scaling inputs
to [-1, 1] helps the network train faster and more stably.


In [2]:
# ============================================================
# STEP 2: Define the Neural Network
# ============================================================

class PINN_Burgers(nn.Module):
    """
    PINN for the Burgers' equation.
    
    Input:  (x, t) — 2 values
    Output: u(x,t) — 1 value (velocity)
    
    Improvements over Project 1:
    - 2D input (x, t)
    - Larger network (4 hidden layers, 64 neurons each)
    - Input normalization for better training
    """
    
    def __init__(self, n_hidden=4, n_neurons=64):
        super().__init__()
        
        layers = []
        
        # Input layer: 2 inputs (x, t) -> n_neurons
        layers.append(nn.Linear(2, n_neurons))
        layers.append(nn.Tanh())
        
        # Hidden layers
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())
        
        # Output layer: n_neurons -> 1 (u)
        layers.append(nn.Linear(n_neurons, 1))
        
        self.network = nn.Sequential(*layers)
        
        # Initialize weights using Xavier initialization
        # This helps PINNs converge faster
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Xavier initialization — better starting point for tanh networks."""
        for m in self.network:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x, t):
        """
        Forward pass.
        
        Args:
            x: Spatial coordinate, shape (N, 1)
            t: Time coordinate, shape (N, 1)
        Returns:
            u: Predicted velocity, shape (N, 1)
        """
        # Concatenate x and t into a single input tensor
        inputs = torch.cat([x, t], dim=1)  # Shape: (N, 2)
        return self.network(inputs)

# Create model
model = PINN_Burgers(n_hidden=4, n_neurons=64).to(device)
print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Model Architecture:
PINN_Burgers(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): Tanh()
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

Total parameters: 12,737


## Step 3: Physics Loss for Burgers' Equation

The PDE residual is:
$$\mathcal{R} = \frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} - \nu \frac{\partial^2 u}{\partial x^2}$$

We need to compute three derivatives:
1. $\partial u / \partial t$ — first derivative w.r.t. time
2. $\partial u / \partial x$ — first derivative w.r.t. space
3. $\partial^2 u / \partial x^2$ — second derivative w.r.t. space

All computed automatically with `torch.autograd.grad`!


In [3]:
# ============================================================
# STEP 3: Physics Loss
# ============================================================

# Physical parameter
nu = 0.01 / np.pi  # Viscosity coefficient

def compute_pde_residual(model, x, t):
    """
    Compute the Burgers' equation residual:
        du/dt + u * du/dx - nu * d²u/dx² = 0
    
    Args:
        model: PINN model
        x: Spatial points, shape (N, 1), requires_grad=True
        t: Time points, shape (N, 1), requires_grad=True
    
    Returns:
        residual: PDE residual at each point
    """
    # Forward pass
    u = model(x, t)
    
    # Compute du/dt (partial derivative w.r.t. time)
    du_dt = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    
    # Compute du/dx (partial derivative w.r.t. space)
    du_dx = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    
    # Compute d²u/dx² (second partial derivative w.r.t. space)
    d2u_dx2 = torch.autograd.grad(
        du_dx, x,
        grad_outputs=torch.ones_like(du_dx),
        create_graph=True
    )[0]
    
    # PDE residual: du/dt + u * du/dx - nu * d²u/dx²
    # This should equal 0 if the PDE is satisfied
    residual = du_dt + u * du_dx - nu * d2u_dx2
    
    return residual


def compute_loss(model, x_pde, t_pde, x_ic, t_ic, u_ic, x_bc, t_bc, u_bc):
    """
    Total PINN loss = PDE loss + Initial Condition loss + Boundary Condition loss.
    
    Three components:
    1. PDE loss: Burgers' equation satisfied at interior collocation points
    2. IC loss:  u(x, 0) = -sin(πx) at initial time
    3. BC loss:  u(-1, t) = u(1, t) = 0 at boundaries
    """
    # 1. PDE Loss (interior points)
    residual = compute_pde_residual(model, x_pde, t_pde)
    loss_pde = torch.mean(residual ** 2)
    
    # 2. Initial Condition Loss: u(x, 0) = -sin(πx)
    u_pred_ic = model(x_ic, t_ic)
    loss_ic = torch.mean((u_pred_ic - u_ic) ** 2)
    
    # 3. Boundary Condition Loss: u(-1, t) = u(1, t) = 0
    u_pred_bc = model(x_bc, t_bc)
    loss_bc = torch.mean((u_pred_bc - u_bc) ** 2)
    
    # Weighted sum — BC and IC get higher weight for stricter enforcement
    w_pde = 1.0
    w_ic = 20.0   # Initial condition is crucial for time-dependent problems
    w_bc = 10.0
    
    total_loss = w_pde * loss_pde + w_ic * loss_ic + w_bc * loss_bc
    
    return total_loss, loss_pde, loss_ic, loss_bc

print("Loss functions defined!")


Loss functions defined!


In [4]:
# ============================================================
# STEP 4: Prepare Training Points
# ============================================================

N_pde = 10000   # Collocation points inside the domain (more needed for 2D)
N_ic = 200      # Initial condition points
N_bc = 200      # Boundary condition points

# --- PDE Collocation Points (interior) ---
# Random points in the domain: x ∈ [-1, 1], t ∈ [0, 1]
x_pde = (2 * torch.rand(N_pde, 1) - 1).to(device).requires_grad_(True)   # [-1, 1]
t_pde = torch.rand(N_pde, 1).to(device).requires_grad_(True)              # [0, 1]

# --- Initial Condition Points: u(x, 0) = -sin(πx) ---
x_ic = (2 * torch.rand(N_ic, 1) - 1).to(device)   # Random x in [-1, 1]
t_ic = torch.zeros(N_ic, 1).to(device)              # t = 0
u_ic = -torch.sin(np.pi * x_ic)                     # Known initial values

# --- Boundary Condition Points: u(-1, t) = u(1, t) = 0 ---
t_bc_vals = torch.rand(N_bc, 1).to(device)          # Random times in [0, 1]

# Left boundary: x = -1
x_bc_left = -torch.ones(N_bc // 2, 1).to(device)
t_bc_left = t_bc_vals[:N_bc // 2]
u_bc_left = torch.zeros(N_bc // 2, 1).to(device)

# Right boundary: x = 1
x_bc_right = torch.ones(N_bc // 2, 1).to(device)
t_bc_right = t_bc_vals[N_bc // 2:]
u_bc_right = torch.zeros(N_bc // 2, 1).to(device)

# Combine boundary points
x_bc = torch.cat([x_bc_left, x_bc_right], dim=0)
t_bc = torch.cat([t_bc_left, t_bc_right], dim=0)
u_bc = torch.cat([u_bc_left, u_bc_right], dim=0)

print(f"PDE points:     {N_pde}")
print(f"IC points:      {N_ic}")
print(f"BC points:      {N_bc}")
print(f"Total points:   {N_pde + N_ic + N_bc}")


PDE points:     10000
IC points:      200
BC points:      200
Total points:   10400


In [ ]:
# ============================================================
# STEP 5: Training Loop
# ============================================================

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

n_epochs = 10000
print_every = 1000

history = {'total': [], 'pde': [], 'ic': [], 'bc': []}

print("Starting training...")
print("=" * 70)

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Resample PDE collocation points each epoch
    x_pde_train = (2 * torch.rand(N_pde, 1) - 1).to(device).requires_grad_(True)
    t_pde_train = torch.rand(N_pde, 1).to(device).requires_grad_(True)
    
    # Compute loss
    total_loss, pde_loss, ic_loss, bc_loss = compute_loss(
        model, x_pde_train, t_pde_train, x_ic, t_ic, u_ic, x_bc, t_bc, u_bc
    )
    
    # Backpropagation and weight update
    total_loss.backward()
    optimizer.step()
    scheduler.step()
    
    # Record losses
    history['total'].append(total_loss.item())
    history['pde'].append(pde_loss.item())
    history['ic'].append(ic_loss.item())
    history['bc'].append(bc_loss.item())
    
    if epoch % print_every == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:5d}/{n_epochs} | "
              f"Total: {total_loss.item():.2e} | "
              f"PDE: {pde_loss.item():.2e} | "
              f"IC: {ic_loss.item():.2e} | "
              f"BC: {bc_loss.item():.2e} | "
              f"LR: {lr:.1e}")

print("=" * 70)
print("Training complete!")


Starting training...
Epoch     1/10000 | Total: 1.17e+01 | PDE: 1.55e-04 | IC: 5.84e-01 | BC: 6.39e-03 | LR: 1.0e-03


In [ ]:
# ============================================================
# STEP 6: Visualize Results
# ============================================================

# Create a grid for plotting
nx, nt = 200, 100
x_grid = np.linspace(-1, 1, nx)
t_grid = np.linspace(0, 1, nt)
X, T = np.meshgrid(x_grid, t_grid)

# Predict on the grid
x_flat = torch.tensor(X.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)
t_flat = torch.tensor(T.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)

model.eval()
with torch.no_grad():
    u_pred = model(x_flat, t_flat).cpu().numpy().reshape(nt, nx)

# --- Figure 1: Solution Contour Plot ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Contour plot of u(x, t)
c1 = axes[0].contourf(X, T, u_pred, levels=50, cmap='RdBu_r')
plt.colorbar(c1, ax=axes[0])
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('t', fontsize=12)
axes[0].set_title('PINN Prediction u(x, t)', fontsize=14)

# Snapshots at different times
time_slices = [0.0, 0.25, 0.5, 0.75, 1.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_slices)))

for i, t_val in enumerate(time_slices):
    t_idx = int(t_val * (nt - 1))
    axes[1].plot(x_grid, u_pred[t_idx, :], color=colors[i], 
                 linewidth=2, label=f't = {t_val:.2f}')

axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('u(x, t)', fontsize=12)
axes[1].set_title('Solution at Different Times', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Training loss history
axes[2].semilogy(history['total'], label='Total', alpha=0.8)
axes[2].semilogy(history['pde'], label='PDE', alpha=0.8)
axes[2].semilogy(history['ic'], label='IC', alpha=0.8)
axes[2].semilogy(history['bc'], label='BC', alpha=0.8)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Loss', fontsize=12)
axes[2].set_title('Training Loss History', fontsize=14)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Key Takeaways

1. **2D Input**: The network learns a function $u(x, t)$ — a 2D surface, not just a curve.

2. **Three loss components**:
   - PDE loss: enforces Burgers' equation at interior collocation points
   - IC loss: enforces the initial condition $u(x, 0) = -\sin(\pi x)$
   - BC loss: enforces boundary conditions $u(\pm 1, t) = 0$

3. **Nonlinearity**: The $u \cdot \partial u / \partial x$ term creates a **shock wave** — 
   the solution develops a sharp front that's challenging for the network.

4. **More points needed**: 2D problems need more collocation points (10,000 vs 100).

5. **Loss weighting matters**: Initial conditions need higher weight because they
   define the entire time evolution.

